# Facilitator-Member Difference Identification

In [ ]:
import json, pandas as pd
from pathlib import Path

# ---- CONFIG ----
DATA_DIR = Path("gemini_data_analysis/data")  # change if needed
OUTPUT_DIR = Path("outputs/person_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOADERS ----
def load_conference_data(conf_path: Path):
    conf_name = conf_path.name   # e.g., "2021MZT"
    year = int(conf_name[:4])
    conference = conf_name[4:]

    def _load(fp):
        with open(fp, "r") as f:
            return json.load(f)

    outcome = _load(conf_path / f"{conf_name}_outcome.json")                 # not used below, but loaded if you need it
    person_to_team = _load(conf_path / f"{conf_name}_person_to_team.json")   # dict: person -> [{team_id, funded_status, ...}]
    session_outcomes = _load(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (per session)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")            # e.g., "2021_09_30_MZT_S5"
        features[sid] = _load(fp)

    return year, conference, outcome, person_to_team, session_outcomes, features

# ---- BUILDERS ----
def build_person_session(year, conference, session_outcomes, features):
    """One row per (person, session), with role_in_session and ctx_* features broadcast from session."""
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]
    rows = []

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []))
        speakers     = set(so.get("all_speakers", []))

        # union of all team members listed under 'teams'
        members = set()
        for _, tinfo in so.get("teams", {}).items():
            for m in tinfo.get("members", []):
                members.add(m)

        people = facilitators | speakers | members

        # session-level features (context)
        ctx = {f"ctx_{k}": v for k, v in features.get(sid, {}).items()}

        for person in sorted(people):
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            rows.append({
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx
            })

    return pd.DataFrame(rows)

def build_person_year(person_session_df, person_to_team):
    """Aggregate to one row per (person, conference, year), add role tallies + team funded counts, and mean ctx_*."""
    if person_session_df.empty:
        return person_session_df

    # Role tallies
    pivot = (person_session_df
             .pivot_table(index=["person_name","conference","year"],
                          columns="role_in_session",
                          values="session_id",
                          aggfunc="nunique",
                          fill_value=0)
             .reset_index())

    for col in ["facilitator","member","participant","unknown"]:
        if col not in pivot.columns:
            pivot[col] = 0

    pivot["sessions_total"] = pivot["facilitator"] + pivot["member"] + pivot["participant"] + pivot["unknown"]

    # Primary role: facilitator > member > participant > unknown
    role_priority = {"facilitator": 3, "member": 2, "participant": 1, "unknown": 0}
    def primary_role(row):
        return max(role_priority, key=lambda r: role_priority[r] * row.get(r, 0))

    pivot["role_primary"] = pivot.apply(primary_role, axis=1)

    # Mean of ctx_* features across sessions the person attended
    ctx_cols = [c for c in person_session_df.columns if c.startswith("ctx_")]
    if ctx_cols:
        ctx_means = (person_session_df
                     .groupby(["person_name","conference","year"], as_index=False)[ctx_cols]
                     .mean())
        out = pivot.merge(ctx_means, on=["person_name","conference","year"], how="left")
    else:
        out = pivot

    # Team outcomes from person_to_team
    team_rows = []
    for pname in out["person_name"]:
        lst = person_to_team.get(pname, [])
        funded = sum(1 for t in lst if t.get("funded_status", 0) == 1)
        unfund = sum(1 for t in lst if t.get("funded_status", 0) == 0)
        team_rows.append((pname, funded, unfund, ", ".join([t.get("team_id","") for t in lst])))
    team_df = pd.DataFrame(team_rows, columns=["person_name","teams_funded","teams_unfunded","team_ids"])
    team_df["teams_total"] = team_df["teams_funded"] + team_df["teams_unfunded"]

    out = out.merge(team_df, on="person_name", how="left")

    return out

# ---- DRIVER ----
for conf_path in DATA_DIR.iterdir():
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    # require the core files to exist
    if not (conf_path / f"{conf_name}_session_outcomes.json").exists():
        continue

    year, conference, outcome, person_to_team, session_outcomes, features = load_conference_data(conf_path)

    person_session_df = build_person_session(year, conference, session_outcomes, features)
    person_year_df    = build_person_year(person_session_df, person_to_team)

    person_session_df.to_csv(OUTPUT_DIR / f"{year}_{conference}_person_session.csv", index=False)
    person_year_df.to_csv(OUTPUT_DIR / f"{year}_{conference}_person_year.csv", index=False)

    print(f"Wrote: {year}_{conference}_person_session.csv ({len(person_session_df)} rows)")
    print(f"Wrote: {year}_{conference}_person_year.csv ({len(person_year_df)} rows)")